In [1]:
import rasterio as rio
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime
import json
import os
from shapely.geometry import box
from rasterio.mask import mask

In [5]:
for image in os.listdir(s2_dir):
    print(image)
    s2_image_path = os.path.join(s2_dir, image)
    print(s2_image_path)

sentinel2_images_mean_2019-02-01_to_2019-03-01-0000006912-0000000000-020.tif
C:\Users\Kostas\Downloads\s2_images\sentinel2_images_mean_2019-02-01_to_2019-03-01-0000006912-0000000000-020.tif
sentinel2_images_mean_2019-06-01_to_2019-07-01-0000006912-0000000000-013.tif
C:\Users\Kostas\Downloads\s2_images\sentinel2_images_mean_2019-06-01_to_2019-07-01-0000006912-0000000000-013.tif


In [2]:
s2_dir = r"E:\S2Germany\S2_images_Germany_monthly_2019"
output_dir = Path(r'E:\S2Germany\Patches')
envelopes_gdf_OG = gpd.read_file(r"C:\Users\Kostas\Desktop\GIMA\Module_7\Data\PEP725\After_2016_sent_from_PEP725\pep725_outputs\PEP725_envelopes.geojson")
envelopes_gdf_OG.set_crs(32632, inplace=True, allow_override=True)

for image in os.listdir(s2_dir):
    print(image)
    s2_image_path = os.path.join(s2_dir, image)
    envelopes_gdf = envelopes_gdf_OG
    with rio.open(s2_image_path) as src:
        print(src.bounds)
        print(src.crs)
        print(src.count)
        src.close
    
    # Separate the gdfs by year
    #envelopes_gdf_2019 = envelopes_gdf[envelopes_gdf['year'] == 2019]
    #envelopes_gdf_2020 = envelopes_gdf[envelopes_gdf['year'] == 2020]
    #envelopes_gdf_2019.head()
    ## 1. Temporal filter
    #Now that everything is loaded the temporal filter should be applied
    # Function to extract the dates from the filename of GEE S2 images

    def imageNamingGEEfiles(raster_path):
        # Example file name: sentinel2_images_mean_2019-07-01_to_2019-08-01-0000006912-0000006912.tif
        string_parts = raster_path.split("_")
        start_date = string_parts[3]
        token = string_parts[5]
        token_string_parts = token.split("-")
        end_date = token_string_parts[0] + "-" + token_string_parts[1] + "-" + token_string_parts[2]
        # Save the month and year to variables
        s2month = datetime.strptime(start_date, '%Y-%m-%d').month
        s2year = datetime.strptime(start_date, '%Y-%m-%d').year
        return start_date, end_date, s2month, s2year

    # Get the start and end dates of the image from its name
    s2_image_start_date, s2_image_end_date, s2month, s2year = imageNamingGEEfiles(Path(s2_image_path).name)
    print(f"Image details \nStart date: {s2_image_start_date}\nEnd date: {s2_image_end_date}\nMonth: {s2month}\nYear: {s2year}")
    
    # Datetime operations in order to do date comparisons and find all the dates that are represented in an image
    # Converting the date column to datetime data type

    envelopes_gdf['date'] = pd.to_datetime(envelopes_gdf['date'], format='%Y-%m-%d').dt.date
    # Converting the outputs to datetime.date dtype
    s2_image_start_date = datetime.strptime(s2_image_start_date, '%Y-%m-%d').date()
    s2_image_end_date = datetime.strptime(s2_image_end_date, '%Y-%m-%d').date()
    # Creating a mask to filter the dates that are needed
    temporal_mask = (envelopes_gdf.date > s2_image_start_date) & (envelopes_gdf.date < s2_image_end_date)
    #display(envelopes_gdf.loc[temporal_mask])
    s2_image_gdf = envelopes_gdf.loc[temporal_mask]
    ## Target extraction
    def addMonths(gdf):
        # Convert the date to datetime type to work later
        gdf['date'] = pd.to_datetime(gdf['date'])

        # Create a Series with the month (1-12)
        # It finds the month (int 1-12) based on the .month method of the datetime property
        # It achieves that by mapping a lambda function on each element of the date column. Therefore the result is just the month number

        getmonth = gdf['date'].map(lambda x:x.month)
        # Another way
        # test_df = test_df.assign(month=test_df['date'].map(lambda x: x.month))

        # Merge this into the gdf
        gdf = gdf.merge(getmonth, left_index=True, right_index=True)

        # Rename the column
        gdf.rename(columns = {'date_y':'month'}, inplace = True)
        return gdf
    envelopes_gdf = addMonths(envelopes_gdf)

    # Group observations by s_id and month, and calculate the label with the maximum frequency for each group
    freqresults_df = envelopes_gdf.groupby(['s_id', pd.Grouper(key='month'), pd.Grouper(key='year')])\
        .apply(lambda x: pd.Series({'Label': x['Label'].value_counts().index[0],
                                    'phase_id': x['phase_id'].value_counts().index[0]}))\
        .reset_index()

    # Time taken: 2m 13.6s
    # Rename the column with the labels
    freqresults_df = freqresults_df.rename(columns={'Label': 'max_label', 'phase_id': 'max_phase_id'})

    freqresults_df.head()
    freqresults_df['max_label'].value_counts()
    freqresults_df['max_phase_id'].value_counts()
    
    ## Spatial filter
    s2_image_gdf = addMonths(s2_image_gdf)
    # Convert the envelopes_gdf to a list to work with the functions
    s2_image_gdf_list = s2_image_gdf.geometry.tolist()
    # This is used to save the indices and then extract the targets directly from the gdf
    s2_image_gdf_index_list = s2_image_gdf.index.values.tolist()
    
    
    """Function to parse features from GeoDataFrame in such a manner that rasterio wants them"""

    def getFeatures(gdf):
            return [json.loads(gdf.to_json())['features'][0]['geometry']]
    '''
    This function reads the envelope list and a raster, checks if the polygons are fully contained in the raster 
    and returns 5 lists, 4 with the boundary coordinates for all the envelopes that are fully contained in the raster 
    and one of their indexes from the full_index_list.
    '''

    def getContainedEnvelopeCoords (raster, envelope_list, full_index_list):
        with rio.open(raster, driver='GTiff') as src:
            raster_extent = src.bounds
            
            # List initialization
            minx_list = []
            miny_list = []
            maxx_list = []
            maxy_list = []
            index_list = []
            for i in range(0, len(envelope_list)):
                poly_extent = envelope_list[i].bounds
                # poly_extent is minx, miny, maxx, maxy

                # Check if the polygon is fully inside the raster's extent
                if (poly_extent[0] > raster_extent[0] and poly_extent[2] < raster_extent[2] and
                    poly_extent[1] > raster_extent[1] and poly_extent[3] < raster_extent[3]):
                        minx_list.append(poly_extent[0])
                        miny_list.append(poly_extent[1])
                        maxx_list.append(poly_extent[2])
                        maxy_list.append(poly_extent[3])
                        index_list.append(full_index_list[i])
        return minx_list, miny_list, maxx_list, maxy_list, index_list

    '''
    This function receives a raster file (.tif) and the boundary coordinates for a polygon. 
    It then clips the raster to the extent of the polygon. 
    The polygon has to intersect the raster for the operation to be completed
    '''

    def exportImage(raster, output_path, minx, miny, maxx, maxy):
        # open the raster file (Single Band)
        data = rio.open(raster, driver='GTiff')

        # Create a bounding box from the polygon min-max coordinates    
        bbox = box(minx, miny, maxx, maxy)
        # Create a geodataframe with a single polygon so that it can be used with rasterio
        geo = gpd.GeoDataFrame({'geometry': bbox}, index=[0], crs='32632')
        # Transform the geodataframe to a GeoJSON-like object that can be used as an input in the rasterio mask function
        coords = getFeatures(geo)
        #print(coords)
        
        # Mask and crop the raster AOI where polygon overlaps the whole raster
        out_img, out_transform = mask(data, shapes=coords, crop=True)
        # Define resolution and more
        out_profile = data.profile.copy()
        
        out_profile.update({'driver':'GTiff', 'width': out_img.shape[2],'height': out_img.shape[1], 'transform': out_transform})
        
        # Write the extracted raster patch to a file
        with rio.open(output_path, 'w', **out_profile) as dst:
            dst.write(out_img)
        
        # data.close()
        # data = None
    minx_list, miny_list, maxx_list, maxy_list, contained_index_list = getContainedEnvelopeCoords(s2_image_path, s2_image_gdf_list, s2_image_gdf_index_list)
    # Create a list with the contained s_id's. It works
    contained_s_id_list = []
    for i in range(0, len(contained_index_list)):
        sid = s2_image_gdf.loc[contained_index_list[i], 's_id']
        contained_s_id_list.append(sid)
    ## Combination of everything to mine the labels
    
    for station in contained_s_id_list:
        condition = (freqresults_df['year'] == s2year) & (freqresults_df['month'] == s2month) & (freqresults_df['s_id'] == station)
        label = freqresults_df.loc[condition, 'max_label'].values[0]
        phase_id = freqresults_df.loc[condition, 'max_phase_id'].values[0]
        print(f'Station with ID {station}, for the year {s2year} and month {s2month} has the label {label} and phenophase with id {phase_id}')
    
    # Create lists of the maximum frequency labels and s_ids to organise the outputs to folders
    unique_labels = freqresults_df['max_label'].unique().tolist()
    unique_phase_ids = freqresults_df['max_phase_id'].unique().tolist()

    # Create folder from one of the lists. Change unique_phase_ids with unique_labels depending on what you want.
    for folder in unique_phase_ids:
        p = Path(output_dir) / str(folder)
        path_exists = Path.exists(p)
        if path_exists:
            print(f'Folder {folder} already exists, skipping...')
        else:
            print(f'Folder {folder} does not exist, creating it...')
            p.mkdir(parents=True, exist_ok=True)

    print("Creating patches for the image: ", Path(s2_image_path).name)

    # Iterating over each envelope in the gdf
    # Reminder, minx_list, contained_index_list and contained_s_id_list have the same length with the same sequence.
    for i in range(0, len(minx_list)):

        # Condition to extract the max_label and max_phase_id from the freqresults_df, based on the s_id of the contained envelope
        condition = (freqresults_df['year'] == s2year) & (freqresults_df['month'] == s2month) & (freqresults_df['s_id'] == contained_s_id_list[i])

        # Get the index and station id for the station with the index in the i-th position
        station_id = contained_s_id_list[i]
        index = contained_index_list[i]

        # Get the maximum frequency label and phase_id for the current envelope based on the month and year
        label = freqresults_df.loc[condition, 'max_label'].values[0]
        phase_id = freqresults_df.loc[condition, 'max_phase_id'].values[0]
        output_folder = str(phase_id)

        # Include the aforementioned information in the image name
        output_name = os.path.join(os.path.join(output_dir, output_folder), Path(s2_image_path).stem + f'index_{index}_station_{station_id}_label_{label}_phase_id_{phase_id}.tif')
        print(f"\t Patch {i+1} out of {len(minx_list) + 1}")

        # Export the patches
        exportImage(s2_image_path, output_name, minx_list[i], miny_list[i], maxx_list[i], maxy_list[i])
    print('Patch creation completed!') 

sentinel2_images_mean_2019-01-01_to_2019-02-01-0000000000-0000000000.tif
BoundingBox(left=280320.0, bottom=5686440.0, right=695040.0, top=6101160.0)
EPSG:32632
23
Image details 
Start date: 2019-01-01
End date: 2019-02-01
Month: 1
Year: 2019


c:\Users\Kostas\anaconda3\envs\pep725\lib\site-packages\geopandas\geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Station with ID 1524, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 1072, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 1296, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 19544, for the year 2019 and month 1 has the label M and phenophase with id 182
Station with ID 461, for the year 2019 and month 1 has the label M and phenophase with id 182
Station with ID 1217, for the year 2019 and month 1 has the label M and phenophase with id 182
Folder 60 does not exist, creating it...
Folder 11 does not exist, creating it...
Folder 131 does not exist, creating it...
Folder 286 does not exist, creating it...
Folder 205 does not exist, creating it...
Folder 95 does not exist, creating it...
Folder 182 does not exist, creating it...
Folder 111 does not exist, creating it...
Folder 10 does not exist, creating it...
Creating patches for the image:  sentinel2_images_mea

c:\Users\Kostas\anaconda3\envs\pep725\lib\site-packages\geopandas\geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Station with ID 5213, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 19541, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 5845, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Station with ID 5363, for the year 2019 and month 1 has the label DBL and phenophase with id 60
Folder 60 already exists, skipping...
Folder 11 already exists, skipping...
Folder 131 already exists, skipping...
Folder 286 already exists, skipping...
Folder 205 already exists, skipping...
Folder 95 already exists, skipping...
Folder 182 already exists, skipping...
Folder 111 already exists, skipping...
Folder 10 already exists, skipping...
Creating patches for the image:  sentinel2_images_mean_2019-01-01_to_2019-02-01-0000000000-0000006912.tif
	 Patch 1 out of 5
	 Patch 2 out of 5
	 Patch 3 out of 5
	 Patch 4 out of 5
Patch creation completed!
sentinel2_images_mean_2019-01-01_to_2019-02-01-0000006912-00

...

c:\Users\Kostas\anaconda3\envs\pep725\lib\site-packages\geopandas\geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Station with ID 3705, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3751, for the year 2019 and month 11 has the label DBL and phenophase with id 95
Station with ID 3792, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3915, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3982, for the year 2019 and month 11 has the label DBL and phenophase with id 95
Station with ID 3986, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3538, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3693, for the year 2019 and month 11 has the label DBL and phenophase with id 205
Station with ID 3849, for the year 2019 and month 11 has the label DBL and phenophase with id 95
Station with ID 3981, for the year 2019 and month 11 has the label DBL and phenophase with id 95
Station with ID 5364, fo

c:\Users\Kostas\anaconda3\envs\pep725\lib\site-packages\geopandas\geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Folder 60 already exists, skipping...
Folder 11 already exists, skipping...
Folder 131 already exists, skipping...
Folder 286 already exists, skipping...
Folder 205 already exists, skipping...
Folder 95 already exists, skipping...
Folder 182 already exists, skipping...
Folder 111 already exists, skipping...
Folder 10 already exists, skipping...
Creating patches for the image:  sentinel2_images_mean_2019-11-01_to_2019-12-01-0000013824-0000000000.tif
Patch creation completed!
sentinel2_images_mean_2019-11-01_to_2019-12-01-0000013824-0000006912.tif
BoundingBox(left=695040.0, bottom=5236440.0, right=921420.0, top=5271720.0)
EPSG:32632
23
Image details 
Start date: 2019-11-01
End date: 2019-12-01
Month: 11
Year: 2019
Folder 60 already exists, skipping...
Folder 11 already exists, skipping...
Folder 131 already exists, skipping...
Folder 286 already exists, skipping...
Folder 205 already exists, skipping...
Folder 95 already exists, skipping...
Folder 182 already exists, skipping...
Folder 1

c:\Users\Kostas\anaconda3\envs\pep725\lib\site-packages\geopandas\geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
